<a href="https://colab.research.google.com/github/aerofa45/PINN_Thesis/blob/main/Guided_PINN_Groundwater_All_Phases_Colab_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# From Poisson to groundwater: guided PINN reliability research lab
### Six-phase exploratory study with **original source code, recorded result tables, and saved plots**
**Status:** teaching/research reproduction notebook. Plots/tables are archived observations from previous runs, **not newly generated by opening this notebook**. Running a phase creates fresh outputs and may differ by hardware/software.

**Use:** (1) Open in Google Colab. (2) Run the setup cell. (3) Upload `research_evidence.zip` from the companion package when prompted. (4) Run the *archive inspection* cells to see all recorded data. (5) Run any or all experiment cells (slower).

The archive includes the untouched original Python source for each phase and the recorded evidence. The groundwater contrast code is the **original .ipynb**, not a recreated approximation.

## 0. Environment and source provenance
- Python; PyTorch / Adam / autograd, NumPy, pandas, matplotlib, SciPy.
- Uploaded starting survey: Cuomo et al. (2022), [DOI](https://doi.org/10.1007/s10915-022-01939-z), especially §5.
- Related method context: Wang et al. (2021), [DOI](https://doi.org/10.1137/20M1318043); Esmaeilbeigi et al. (2026), [DOI](https://doi.org/10.1007/s10915-026-03447-w).
- We used a **custom gradient-norm-ratio heuristic**, not a reproduction of a named state-of-the-art weighting algorithm.
- **Synthetic** 1D groundwater model, not field observations.


In [ ]:
%pip -q install torch numpy pandas matplotlib scipy nbformat
from pathlib import Path
import os, zipfile, math, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown
ROOT=Path('/content/pinn_study')
ROOT.mkdir(parents=True,exist_ok=True)
print('Working directory:',ROOT)
print('Upload research_evidence.zip from the companion downloaded package.')
from google.colab import files
uploaded=files.upload()
assert 'research_evidence.zip' in uploaded, 'Select research_evidence.zip'
with zipfile.ZipFile('research_evidence.zip') as z:
    z.extractall(ROOT)
print('Imported:',len(list(ROOT.rglob('*'))),'paths')

## 1. Physics model and evaluation criteria
**Poisson:** $-u''(x)=\pi^2\sin(\pi x)+0.2(3\pi)^2\sin(3\pi x)$, with $u(0)=u(1)=0$, known solution $u=\sin(\pi x)+0.2\sin(3\pi x)$.

**Groundwater:** $-(k(x)h'(x))'=1$, $h(0)=1$, $h(1)=0$. $k(x)=\exp[-\log(C)\,\sigma((x-0.5)/0.04)]$, $C=1,10,100$. A smooth transition (NOT a discontinuous interface). Darcy flux $q=-kh'$, hence $q'=1$.

We report relative head $L_2$ error, flux-conservation RMSE of $q'-1$, boundary mismatch, gradient cosine (positive cooperative / negative opposing), and occasionally actual flux-profile error. We do **not** claim theoretical convergence from these measurements.

## 2. Recorded evidence: view plots and tables before running expensive training
**Source of every archived figure and CSV:** results bundled with the earlier completed local experiments. This view is instant and reproducible without a GPU. Results from fresh reruns will be written to a different folder.

### Phase 1 — Smooth Poisson pilot: fixed vs adaptive
12 seeds, 800 steps. Adaptive median L2 0.000142 vs fixed-1 0.001655. This showed promise relative to **one untuned baseline**, not proof of a superior algorithm.

In [ ]:
print('Archived table: results/01_poisson/metrics.csv')
df=pd.read_csv(ROOT/'results/01_poisson/metrics.csv')
display(df.head(30))
print('Rows:',len(df),'Columns:',list(df.columns))

In [ ]:
display(Image(filename=str(ROOT/'results/01_poisson/training_error.png'),width=920))

In [ ]:
display(Image(filename=str(ROOT/'results/01_poisson/solution.png'),width=920))

**Original source code** (display line-by-line or run the original file, with outputs saved separately):

In [ ]:
source=ROOT/'code/01_poisson_pilot.py'
print('ORIGINAL CODE:',source,'\n')
print(source.read_text()[:11000])
print('Total lines:',len(source.read_text().splitlines()))

### Phase 2 — Groundwater conductivity contrasts
18 runs (C=1,10,100; three seeds, two methods). At C=100, fixed and adaptive medians ≈ 0.1424 and 0.1404 after 1200 epochs. A more demanding heterogeneous configuration exposed training difficulties.

In [ ]:
print('Archived table: results/02_groundwater/metrics.csv')
df=pd.read_csv(ROOT/'results/02_groundwater/metrics.csv')
display(df.head(30))
print('Rows:',len(df),'Columns:',list(df.columns))

In [ ]:
display(Image(filename=str(ROOT/'results/02_groundwater/error_vs_contrast.png'),width=920))

In [ ]:
display(Image(filename=str(ROOT/'results/02_groundwater/solution_comparison.png'),width=920))

In [ ]:
display(Image(filename=str(ROOT/'results/02_groundwater/reference_profiles.png'),width=920))

**Original executable notebook** is included: `notebooks/Project2_Groundwater_PINN_Colab.ipynb`. In Colab select File → Open notebook → Upload to execute its native cells. It contains the full conductivity-contrast experiment.

### Phase 2b — 5000-epoch diagnostic
Two seeds: training longer markedly decreased errors; therefore 1200-epoch poor performance was not evidence of an inherent PINN failure.

In [ ]:
print('Archived table: results/02_groundwater/5000_epoch_diagnostic.csv')
df=pd.read_csv(ROOT/'results/02_groundwater/5000_epoch_diagnostic.csv')
display(df.head(30))
print('Rows:',len(df),'Columns:',list(df.columns))

**Original source code** (display line-by-line or run the original file, with outputs saved separately):

In [ ]:
source=ROOT/'code/03_5000_epoch_diagnostic.py'
print('ORIGINAL CODE:',source,'\n')
print(source.read_text()[:11000])
print('Total lines:',len(source.read_text().splitlines()))

### Phase 3 — Loss-weight sweep and conservation
At C=100 after 2500 steps, fixed lambda=0.1 had median head error 0.002759 and conservation RMSE 0.03421, outperforming the tested adaptive heuristic (head 0.06581, conservation 0.15053).

In [ ]:
print('Archived table: results/03_weight_sweep/summary.csv')
df=pd.read_csv(ROOT/'results/03_weight_sweep/summary.csv')
display(df.head(30))
print('Rows:',len(df),'Columns:',list(df.columns))

In [ ]:
display(Image(filename=str(ROOT/'results/03_weight_sweep/error_convergence.png'),width=920))

In [ ]:
display(Image(filename=str(ROOT/'results/03_weight_sweep/flux_conservation.png'),width=920))

In [ ]:
display(Image(filename=str(ROOT/'results/03_weight_sweep/flux_profiles.png'),width=920))

**Original source code** (display line-by-line or run the original file, with outputs saved separately):

In [ ]:
source=ROOT/'code/04_loss_weight_sweep.py'
print('ORIGINAL CODE:',source,'\n')
print(source.read_text()[:11000])
print('Total lines:',len(source.read_text().splitlines()))

### Phase 4 — Adaptive weight capping
Capping adaptive lambda at 1 reduced median head error to ≈ 0.01667, from ≈ 0.06581 uncapped. Suggestive of overweighting, not conclusive causal isolation.

In [ ]:
print('Archived table: results/04_mechanism/combined_summary.csv')
df=pd.read_csv(ROOT/'results/04_mechanism/combined_summary.csv')
display(df.head(30))
print('Rows:',len(df),'Columns:',list(df.columns))

In [ ]:
display(Image(filename=str(ROOT/'results/04_mechanism/error_vs_epochs.png'),width=920))

In [ ]:
display(Image(filename=str(ROOT/'results/04_mechanism/weight_trajectories.png'),width=920))

**Original source code** (display line-by-line or run the original file, with outputs saved separately):

In [ ]:
source=ROOT/'code/05_adaptive_cap_ablation.py'
print('ORIGINAL CODE:',source,'\n')
print(source.read_text()[:11000])
print('Total lines:',len(source.read_text().splitlines()))

### Phase 5 — Identical checkpoint intervention
Pretrain 500 epochs at weight 1, copy model AND optimizer, branch 1200 steps. Fixed-1 median head error 0.07654; fixed-0.1 0.10057; adaptive 0.09624. Changes made after pretraining behave differently from changes made at initialization.

In [ ]:
print('Archived table: results/05_checkpoint/summary.csv')
df=pd.read_csv(ROOT/'results/05_checkpoint/summary.csv')
display(df.head(30))
print('Rows:',len(df),'Columns:',list(df.columns))

In [ ]:
display(Image(filename=str(ROOT/'results/05_checkpoint/head_error_trajectories.png'),width=920))

In [ ]:
display(Image(filename=str(ROOT/'results/05_checkpoint/gradient_alignment.png'),width=920))

In [ ]:
display(Image(filename=str(ROOT/'results/05_checkpoint/weighted_gradient_ratio.png'),width=920))

**Original source code** (display line-by-line or run the original file, with outputs saved separately):

In [ ]:
source=ROOT/'code/06_identical_checkpoint_intervention.py'
print('ORIGINAL CODE:',source,'\n')
print(source.read_text()[:11000])
print('Total lines:',len(source.read_text().splitlines()))

### Phase 6 — 10-seed timing experiment
Ten seeds, checkpoints 250/1000/2500, 400-step continuation per branch. Remaining at weight 1 had the smallest median head error at all checkpoints; physics conservation ranked strategies differently. Later checkpoints also have larger total budgets: do not interpret raw cross-checkpoint differences as causal timing effects.

In [ ]:
print('Archived table: results/06_timing/summary.csv')
df=pd.read_csv(ROOT/'results/06_timing/summary.csv')
display(df.head(30))
print('Rows:',len(df),'Columns:',list(df.columns))

In [ ]:
display(Image(filename=str(ROOT/'results/06_timing/head_comparison.png'),width=920))

In [ ]:
display(Image(filename=str(ROOT/'results/06_timing/flux_comparison.png'),width=920))

In [ ]:
display(Image(filename=str(ROOT/'results/06_timing/gradient_comparison.png'),width=920))

**Original source code** (display line-by-line or run the original file, with outputs saved separately):

In [ ]:
source=ROOT/'code/07_timing_10seed_study.py'
print('ORIGINAL CODE:',source,'\n')
print(source.read_text()[:11000])
print('Total lines:',len(source.read_text().splitlines()))

## 3. Rerun experiments (optional, potentially slow)
The cells below use the **archived original code**. They do NOT overwrite the archived evidence. The full-scale studies can take minutes or longer depending on Colab compute. Some scripts contain nonstandard CLI argument parsers; their exact command-line behavior is preserved as originally written. The groundwater contrast phase is rerun from its dedicated notebook.

Start with a smaller number of seeds/epochs for a smoke test; use the full settings before reporting scientific results. Do not compare fresh results with archived values without recording environment and code versions.

In [ ]:
import subprocess, sys
RUN=ROOT/'new_runs';RUN.mkdir(exist_ok=True)
def run_original(relative_script, args):
    command=[sys.executable,str(ROOT/relative_script)]+args
    print('RUN:', ' '.join(command))
    return subprocess.run(command,cwd=RUN,check=False)
# Sample: first phase, 2 seeds for a quick smoke test. Inspect script flags first.
print((ROOT/'code/01_poisson_pilot.py').read_text()[-1200:])

In [ ]:
# Phase 1 smoke run. Increase seeds and epochs after the quick check.
run_original('code/01_poisson_pilot.py',['--epochs','30','--seeds','2','--out',str(RUN/'poisson_smoke')])

In [ ]:
# Full Phase 1: 12 seeds × 800 epochs. Uncomment to execute.
# run_original('code/01_poisson_pilot.py',['--epochs','800','--seeds','12','--out',str(RUN/'poisson_full')])

#### Original run — 03_5000_epoch_diagnostic.py
Review `--help` or the source header before running. Default parameters reproduce the design, but some original scripts may use fixed output paths: **use a separate Colab workspace** to avoid mixing results.

In [ ]:
print((ROOT/'code/03_5000_epoch_diagnostic.py').read_text()[-1600:])

In [ ]:
# Uncomment after reading the flags and choosing a fresh output path.
# run_original('code/03_5000_epoch_diagnostic.py', ['--help'])

#### Original run — 04_loss_weight_sweep.py
Review `--help` or the source header before running. Default parameters reproduce the design, but some original scripts may use fixed output paths: **use a separate Colab workspace** to avoid mixing results.

In [ ]:
print((ROOT/'code/04_loss_weight_sweep.py').read_text()[-1600:])

In [ ]:
# Uncomment after reading the flags and choosing a fresh output path.
# run_original('code/04_loss_weight_sweep.py', ['--help'])

#### Original run — 05_adaptive_cap_ablation.py
Review `--help` or the source header before running. Default parameters reproduce the design, but some original scripts may use fixed output paths: **use a separate Colab workspace** to avoid mixing results.

In [ ]:
print((ROOT/'code/05_adaptive_cap_ablation.py').read_text()[-1600:])

In [ ]:
# Uncomment after reading the flags and choosing a fresh output path.
# run_original('code/05_adaptive_cap_ablation.py', ['--help'])

#### Original run — 06_identical_checkpoint_intervention.py
Review `--help` or the source header before running. Default parameters reproduce the design, but some original scripts may use fixed output paths: **use a separate Colab workspace** to avoid mixing results.

In [ ]:
print((ROOT/'code/06_identical_checkpoint_intervention.py').read_text()[-1600:])

In [ ]:
# Uncomment after reading the flags and choosing a fresh output path.
# run_original('code/06_identical_checkpoint_intervention.py', ['--help'])

#### Original run — 07_timing_10seed_study.py
Review `--help` or the source header before running. Default parameters reproduce the design, but some original scripts may use fixed output paths: **use a separate Colab workspace** to avoid mixing results.

In [ ]:
print((ROOT/'code/07_timing_10seed_study.py').read_text()[-1600:])

In [ ]:
# Uncomment after reading the flags and choosing a fresh output path.
# run_original('code/07_timing_10seed_study.py', ['--help'])

## 4. Recompute the summary from the archived measurements
This is an independent check that the displayed timing-study medians are computed from the underlying measurements rather than typed in manually.

In [ ]:
m=pd.read_csv(ROOT/'results/06_timing/measurements.csv')
print('Available columns:',list(m.columns))
final=m[m['step']==m['step'].max()].copy()
print('10-seed timing study: medians from source measurements')
display(final.groupby(['checkpoint','mode'])[['head_l2','flux_cons','cosine']].median().round(6))

## 5. Research interpretation, limitations, and next questions
**Observed:** loss choice, training budget, and optimizer history all change results; predictive and conservation rankings diverge; gradient opposition is frequent but not sufficient to explain performance.

**Not demonstrated:** general convergence proof, universal advantage of weight 0.1, failure of all adaptive optimizers, causal superiority of a particular intervention time, field validity, or original optimization algorithm.

**Next experiments:** (1) matched total epochs and wall time; (2) hard Dirichlet boundary solution $h_\theta=1-x+x(1-x)N_\theta$; (3) tuned fixed schedules and correctly implemented published adaptive methods; (4) sharp material interfaces with flux continuity; (5) 2D and inverse hydraulic conductivity; (6) more seeds and confidence intervals.

**Reading order:** Cuomo et al. (2022) survey §5; Wang et al. (2021) gradient-flow pathologies; Esmaeilbeigi et al. (2026) loss weighting review. Check 2023–2026 literature when asserting novelty.

**Export:** In Colab use File → Download → Download .ipynb. To download all extracted evidence use `!zip -qr research_output.zip /content/pinn_study` then download via the Colab file sidebar.